# Aula 02A · Diferenças finitas

Esta semana apresenta o [capítulo 2 do site](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/). A ideia central: **o
computador não sabe derivar — ele subtrai e divide**. A derivada vira
$\dfrac{f(x+h) - f(x-h)}{2h}$, com um $h$ pequeno mas não nulo. A série de
Taylor diz de onde vem essa fórmula e quanto ela erra.

**Ao fim dos dois encontros você consegue:**

1. deduzir, no papel, as diferenças progressiva, regressiva e central a partir
   da série de Taylor, e dizer a ordem do erro de cada uma;
2. calcular uma derivada numérica em Python e medir o erro contra a exata;
3. **ver** a ordem do erro (cortando $h$ pela metade) e explicar por que um $h$
   pequeno demais piora o resultado;
4. usar a segunda derivada para saber se um crescimento está acelerando ou freando.

**Encontro A — a ideia** (1h40): 🧩 · 1. limite · 2. 🧑‍🏫 Taylor · 3. três
fórmulas · 4. ordem · 5. h pequeno demais · 6. segunda derivada · 7. outra área · 🚪

O encontro B é o laboratório: o problema da maré resolvido e a Lista 02 começada.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da semana

> **Picãozinho, João Pessoa — oceanografia.**
>
> *Os barqueiros que levam turistas às piscinas naturais do Picãozinho dependem
> da maré: o passeio só acontece com a maré baixa, e a volta tem de ser antes de a
> água subir demais. Um deles pergunta: "**no horário dos passeios, entre 6 h e
> 18 h, quando a maré sobe mais depressa, e quantos centímetros por hora ela sobe
> nesse momento?**" A tábua de marés dá o nível; ninguém publica a velocidade.*

"Com que rapidez" é uma derivada. Hoje você aprende a calculá-la sem derivar
nada à mão; no encontro B, responde ao barqueiro.

## 1. A derivada como limite

A derivada é o limite da inclinação da secante, $\dfrac{f(x+h) - f(x)}{h}$,
quando $h \to 0$. O computador não faz limite: ele **para** num $h$ pequeno e
aceita o erro.

📖 [capítulo 2 · A derivada como limite](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#a-derivada-como-limite)

**✍️ Passo 1.** Defina `f(x)` que devolve `x**2`. Com `x = 2` e `h = 1`, calcule e imprima `(f(x + h) - f(x)) / h`.

In [ ]:
# ✍️ passo 1

**Preveja:** a derivada exata de $x^2$ em 2 é 4. A conta vai dar mais, menos ou exatamente 4?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dá `5.0`: a secante entre $x = 2$ e $x = 3$ é mais inclinada que a tangente
em $x = 2$, porque a parábola vai ficando mais íngreme.

</details>

**✍️ Passo 2.** Repita a conta num laço `for h in [1, 0.1, 0.01, 0.001]:`, imprimindo `h` e a inclinação.

In [ ]:
# ✍️ passo 2

**Preveja:** quanto vale o **erro** (inclinação − 4) em cada linha? Que padrão ele segue?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai `5.0`, `4.1`, `4.01`, `4.001` (com alguma sujeira nas últimas casas).
O erro é **igual a $h$**: $h$ dez vezes menor, erro dez vezes menor. Esse
padrão tem nome — é o bloco 4.

📖 [capítulo 2 · A derivada como limite](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#a-derivada-como-limite)

</details>

## 2. No quadro: as fórmulas saem de Taylor

A fórmula do bloco 1 foi tirada da definição. A série de Taylor mostra que existe
outra — **bem melhor** — e diz quanto cada uma erra.

📖 [capítulo 2 · No quadro: as fórmulas saem de Taylor](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#no-quadro-as-formulas-saem-de-taylor)

### 🧑‍🏫 No quadro — dedução das diferenças finitas

Caderno de papel aberto. No quadro:

1. a série de Taylor de $f(x+h)$;
2. isolar $f'(x)$ → **progressiva**, e o que foi jogado fora começa com $h$;
3. a série de $f(x-h)$ → **regressiva**;
4. **subtrair** as duas séries: somem $f(x)$ e $f''(x)$;
5. dividir por $2h$ → **central**, e o que sobra começa com $h^2$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

| Fórmula | Expressão | Erro |
|---|---|---|
| progressiva | $\dfrac{f(x+h) - f(x)}{h}$ | $O(h)$ |
| regressiva | $\dfrac{f(x) - f(x-h)}{h}$ | $O(h)$ |
| central | $\dfrac{f(x+h) - f(x-h)}{2h}$ | $O(h^2)$ |

A central é melhor porque, na subtração do passo 4, o termo com $f''(x)$ — o que
dava o erro proporcional a $h$ — **se cancela**.

</details>

## 3. Três fórmulas, um circuito

Num indutor, a tensão é $v = L\,\dfrac{di}{dt}$: ele reage à **variação** da
corrente. A corrente de um circuito RLC amortecido é
$i(t) = e^{-0{,}5t}\sin(2t)$ A. Vamos derivar em $t = 1$ s com as três fórmulas.

📖 [capítulo 2 · Três fórmulas, um circuito](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#tres-formulas-um-circuito)

**✍️ Passo 3.** Defina `i(t)` que devolve `np.exp(-0.5 * t) * np.sin(2 * t)`. Com `t = 1.0` e `h = 0.1`, calcule `progressiva` e imprima.

In [ ]:
# ✍️ passo 3

**Preveja:** a corrente está aumentando ou diminuindo em $t = 1$ s?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai perto de `-0.85`: negativo, a corrente está **diminuindo**. O seno já
passou do pico ($2t = 2$ rad passou de $\pi/2$).

</details>

**✍️ Passo 4.** Calcule também `regressiva` e `central` (lembre dos parênteses em `(2 * h)`) e imprima as três.

In [ ]:
# ✍️ passo 4

**Preveja:** a central fica entre as outras duas? Mais perto de qual?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`-0.8505`, `-0.6944` e `-0.7725`. A central é **exatamente a média** das
outras duas (faça a conta!). A progressiva erra para um lado, a regressiva
para o outro, e a média compensa.

</details>

> ⚠️ **Armadilha.** `(i(t + h) - i(t - h)) / 2 * h` **não** é a central: o Python divide por
2 e depois **multiplica** por `h`. Sai um número 100 vezes menor, sem
mensagem de erro nenhuma.

**✍️ Passo 5.** A derivada exata (regra do produto) é `np.exp(-0.5*t) * (2*np.cos(2*t) - 0.5*np.sin(2*t))`. Guarde em `exata` e imprima o erro relativo percentual de cada fórmula.

In [ ]:
# ✍️ passo 5

**Preveja:** quantas vezes o erro da central é menor que o da progressiva?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cerca de 9 %, 11 % e **1 %**: a central erra quase nove vezes menos, com o
mesmo $h$ e o mesmo trabalho (duas avaliações de `i`).

</details>

### 🎯 Sua vez — A função central

A partir de agora, a fórmula vira **função**. Escreva `central(f, x, h)`,
que devolve a diferença central de `f` em `x`. O parâmetro `f` é uma
**função** — você a chama dentro de `central`, como `f(x + h)`.

In [ ]:
def central(f, x, h):
    # sua solução aqui
    pass

In [ ]:
def cubo(x):
    return x**3


confere(central, [
    ((cubo, 2, 0.1), 12.010000000000009),
    ((np.sin, 0.0, 0.1), np.float64(0.9983341664682815)),
])

<details>
<summary><b>💡 Dica</b></summary>

É a mesma conta do passo 4, trocando `i` por `f` e devolvendo com `return`.

</details>

## 4. A ordem do erro

O quadro prometeu: progressiva $O(h)$, central $O(h^2)$. Dá para **ver** isso
cortando $h$ pela metade e medindo quanto o erro cai.

📖 [capítulo 2 · A ordem do erro](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#a-ordem-do-erro)

**✍️ Passo 6.** Para $e^x$ em `x = 1.0` (a derivada exata é `np.exp(1.0)`), calcule o erro absoluto da **central** com `h = 0.1` e com `h = 0.05`, e imprima a razão `erro_1 / erro_2`.

In [ ]:
# ✍️ passo 6

**Preveja:** a central é $O(h^2)$. Que número a razão deve dar?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Perto de **4**. Com $h$ pela metade, o erro cai $2^2 = 4$ vezes. O expoente
da ordem aparece na razão.

</details>

**✍️ Passo 7.** Repita o passo 6 com a **progressiva**.

In [ ]:
# ✍️ passo 7

**Preveja:** e agora, que razão?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Perto de **2** ($2^1$). No capítulo, o gráfico log-log mostra as duas como
retas: a inclinação de cada reta é a ordem.

📖 [capítulo 2 · A ordem do erro](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#a-ordem-do-erro)

</details>

## 5. Quando h pequeno demais atrapalha

Se a central melhora 100 vezes a cada $h$ dez vezes menor, por que não usar
$h = 10^{-15}$?

📖 [capítulo 2 · Quando h pequeno demais atrapalha](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#quando-h-pequeno-demais-atrapalha)

**✍️ Passo 8.** Num laço `for k in range(1, 16):`, faça `h = 10.0**(-k)` e imprima `k` e o erro absoluto da central de $e^x$ em `x = 1.0`.

In [ ]:
# ✍️ passo 8

**Preveja:** o erro vai cair até o fim? Em que `k` ele fica menor?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cai até `k = 5` ou `6` (erro perto de $10^{-10}$) e depois **sobe**: em
`k = 15` o erro é pior que em `k = 1`. Com $h$ minúsculo, $f(x+h)$ e
$f(x-h)$ ficam quase iguais, e a subtração apaga os algarismos
significativos. Sobra só o arredondamento, ampliado pela divisão por $2h$.

</details>

> ⚠️ **Armadilha.** "Menor $h$ = mais preciso" só vale **até certo ponto**. Na tela, `h = 1e-15`
parece o mais caprichado e é o pior de todos, sem aviso nenhum.

## 6. A segunda derivada

Somando as duas séries de Taylor (em vez de subtrair), quem se cancela são $f'$ e
$f'''$, e sai

$$ f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}. $$

$f'' > 0$: a variação está **acelerando**. $f'' < 0$: está **freando**.

📖 [capítulo 2 · A segunda derivada](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#a-segunda-derivada)

Uma colônia de bactérias num frasco cresce segundo a **logística** da célula 📦 abaixo: devagar no começo, depressa no meio e freando quando o alimento acaba.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Colônia de bactérias (milhares), t horas depois do início (modelo logístico).
def P(t):
    return 1000 / (1 + 99 * np.exp(-0.5 * t))

**✍️ Passo 9.** Imprima a segunda derivada central de `P` (com `h = 0.01`) em `t = 8` e `t = 10`.

In [ ]:
# ✍️ passo 9

**Preveja:** qual o sinal em cada instante? O que aconteceu entre 8 e 10 horas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Positiva em 8 (≈ 16,6), negativa em 10 (≈ −12). Entre os dois, o
crescimento **parou de acelerar e começou a frear**: é o ponto de inflexão,
a hora de crescimento mais rápido da colônia.

</details>

### 🎯 Sua vez — Acelerando ou freando?

Escreva `fase(t)`, que devolve o texto `"acelerando"` se a segunda derivada
central de `P` em `t` (com `h = 0.01`) for positiva, e `"freando"` caso
contrário. Use a função `P` da célula 📦.

In [ ]:
def fase(t):
    # sua solução aqui
    pass

In [ ]:
confere(fase, [
    ((4,), "acelerando"),
    ((9,), "acelerando"),
    ((10,), "freando"),
    ((12,), "freando"),
])

<details>
<summary><b>💡 Dica</b></summary>

Calcule `d2` com a fórmula do bloco e use um `if d2 > 0:`.

</details>

## 7. Mesmo método, outra área

A sua `central` do 🎯 não sabe nada de circuitos: ela recebe **qualquer** função.
Economia: numa fábrica, o **custo marginal** — quanto custa uma peça a mais — é a
derivada do custo total.

📖 [capítulo 2 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/#mesmo-metodo-outra-area)

**✍️ Passo 10.** Defina `custo(q)` que devolve `5000 + 12*q + 0.01*q**2 + 0.00001*q**3`. Imprima `central(custo, 500, 1)`.

In [ ]:
# ✍️ passo 10

**Preveja:** é preciso mudar alguma coisa em `central` para ela funcionar com custos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Nada: sai perto de `29.5` (reais por peça). O mesmo código deriva corrente, bactéria
ou custo. É isso que "método numérico" quer dizer: a conta não depende da
área.

</details>

## 🚪 Antes de sair

**1.** A central usa $f(x+h)$ e $f(x-h)$. Se a sua função é uma **tabela de
medições** — digamos, a temperatura a cada 5 minutos —, que $h$ você usa? E no
**primeiro** ponto da tabela, onde não existe $f(x-h)$?

> 🌉 **Esta fica sem resposta aqui.** É por ela que o encontro B começa.

**2.** Por que a central erra menos que a progressiva, com o mesmo $h$?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Na dedução por Taylor, a subtração das séries de $f(x+h)$ e $f(x-h)$ **cancela** o
termo com $f''(x)$, que é o que dá o erro proporcional a $h$. O erro que sobra
começa em $h^2$.

</details>

**3.** Você calculou uma derivada com `h = 1e-12` e ela saiu com 4 casas certas.
Com `h = 1e-4` sairiam mais ou menos casas certas?

<details>
<summary><b>▶ Resposta da 3</b></summary>

**Mais.** `1e-12` está do lado "arredondamento" do V: a subtração quase perdeu tudo.
Com `1e-4`, perto do fundo do V, a central de uma função bem-comportada acerta 8 a
10 casas.

</details>

## 🏠 Para casa

- Releia o [capítulo 2](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/), principalmente a caixa 🧑‍🏫 — refaça a
  dedução da central **sem olhar**.
- No encontro B: o problema da maré, a pergunta 🌉 e a [Lista 02](https://lacouth.github.io/metodos_telecom-site/listas/lista02/) começada
  em sala.